# 00 · Setup do catálogo olist e carga da landing

**Cria a estrutura no Unity Catalog** e baixa o dataset original da Olist para o volume `landing.olist_source`.

**Idempotente:** pode ser executado quantas vezes quiser; nada é duplicado e nenhum comando falha por já existir.

**Dependência:** kagglehub==1.0.3 declarada no painel *Environment* do notebook).

In [0]:
%sql
-- Estrutura: catálogo e schemas da arquitetura medalhão
CREATE CATALOG IF NOT EXISTS olist;
CREATE SCHEMA IF NOT EXISTS olist.landing;
CREATE SCHEMA IF NOT EXISTS olist.bronze;
CREATE SCHEMA IF NOT EXISTS olist.silver;
CREATE SCHEMA IF NOT EXISTS olist.gold;
CREATE SCHEMA IF NOT EXISTS olist.quality;
CREATE SCHEMA IF NOT EXISTS olist.ops;

-- Volumes da landing
CREATE VOLUME IF NOT EXISTS olist.landing.olist_source;
CREATE VOLUME IF NOT EXISTS olist.landing.raw_data;

### Comentários da estrutura criada

In [0]:
%sql
-- Documentação dos objetos. COMMENT ON é idempotente e também atualiza objetos já existentes.
COMMENT ON CATALOG olist IS 'Lakehouse do marketplace Olist — projeto de estudo (dataset CC BY-NC-SA, uso educacional)';
COMMENT ON SCHEMA olist.landing IS 'Zona de chegada: arquivos brutos em Volumes, antes de qualquer ingestão';
COMMENT ON SCHEMA olist.bronze  IS 'Dados brutos ingeridos via Auto Loader, append-only, com colunas de auditoria';
COMMENT ON SCHEMA olist.silver  IS 'Dados limpos, tipados, deduplicados; CDC/SCD2 de vendedores e clientes';
COMMENT ON SCHEMA olist.gold    IS 'Modelo dimensional e métricas oficiais para consumo';
COMMENT ON SCHEMA olist.quality IS 'Resultados de qualidade de dados e quarentena de registros inválidos';
COMMENT ON SCHEMA olist.ops     IS 'Operação: logs de execução (run_log) e controle de watermarks';
COMMENT ON VOLUME olist.landing.olist_source IS 'Dataset original da Olist (Kaggle olistbr/brazilian-ecommerce, versão 2). Somente leitura: fonte do simulador';
COMMENT ON VOLUME olist.landing.raw_data     IS 'Arquivos que chegam do simulador, uma subpasta por entidade. Lido pelo Auto Loader';


## Download e Carga dos dados na Landing

In [0]:
# DBTITLE 1,Download do dataset da Olist (Kaggle)
import os
import shutil
import tempfile

import kagglehub

# Versão fixada: garante que toda execução baixe exatamente os mesmos dados
DATASET = "olistbr/brazilian-ecommerce/versions/2"
TARGET_DIR = "/Volumes/olist/landing/olist_source"
EXPECTED_FILES = {
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
}

In [0]:
# Cache do kagglehub no disco local temporário do compute, fora do volume:
# só os arquivos finais vão para a landing e o cache é apagado ao final.
cache_dir = tempfile.mkdtemp(prefix="kagglehub_")
os.environ["KAGGLEHUB_CACHE"] = cache_dir

try:
    path = kagglehub.dataset_download(DATASET)

    missing = EXPECTED_FILES - set(os.listdir(path))
    if missing:
        raise FileNotFoundError(f"Arquivos ausentes no download: {sorted(missing)}")

    for file_name in sorted(EXPECTED_FILES):
        dst = os.path.join(TARGET_DIR, file_name)
        shutil.copyfile(os.path.join(path, file_name), dst)  # sobrescreve: rodar 2x não duplica
        print("Salvo:", dst)
finally:
    shutil.rmtree(cache_dir, ignore_errors=True)

In [0]:

# DBTITLE 1,Validação: contagem de linhas x valores conhecidos do dataset
EXPECTED_ROWS = {
    "olist_customers_dataset.csv": 99_441,
    "olist_geolocation_dataset.csv": 1_000_163,
    "olist_order_items_dataset.csv": 112_650,
    "olist_order_payments_dataset.csv": 103_886,
    "olist_order_reviews_dataset.csv": 99_224,
    "olist_orders_dataset.csv": 99_441,
    "olist_products_dataset.csv": 32_951,
    "olist_sellers_dataset.csv": 3_095,
    "product_category_name_translation.csv": 71,
}

divergentes = []
for file_name, expected in EXPECTED_ROWS.items():
    # multiLine + escape: os comentários das reviews têm quebras de linha e aspas dentro do texto
    actual = (
        spark.read.option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(f"{TARGET_DIR}/{file_name}")
        .count()
    )
    status = "OK" if actual == expected else "DIVERGENTE"
    print(f"{status:<10} {file_name:<42} esperado={expected:>9,}  lido={actual:>9,}")
    if actual != expected:
        divergentes.append(file_name)

assert not divergentes, f"Contagens divergentes: {divergentes}"
